# M5: ?????? ? ???????????? (??? M1)

??????? ?????? ?????? ???????? M5:
1. ?????? ?????? ??????????? ?? ?? ?? ??????.
2. ?????? ???????? M5 ? ??????? ???????.
3. ?????? ????????????.
4. ????????? `CSV` ? `PNG`.


In [ ]:
import io
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests


In [ ]:
def make_session() -> requests.Session:
    s = requests.Session()
    # ????? ?? ???????? ??-?? ?????????? ?????? 127.0.0.1:9
    s.trust_env = False
    s.headers.update({'User-Agent': 'Mozilla/5.0'})
    return s


def flatten_columns(columns: pd.Index) -> list[str]:
    out = []
    for col in columns:
        if isinstance(col, tuple):
            parts = []
            for p in col:
                p = str(p).strip()
                if p and not p.startswith('Unnamed') and p not in parts:
                    parts.append(p)
            out.append(' | '.join(parts))
        else:
            out.append(str(col).strip())
    return out


def to_num(s: pd.Series) -> pd.Series:
    return pd.to_numeric(
        s.astype(str).str.replace(' ', '', regex=False).str.replace(',', '.', regex=False),
        errors='coerce',
    )


def mad_score(series: pd.Series, window: int = 756) -> pd.Series:
    min_p = max(30, window // 10)
    med = series.rolling(window=window, min_periods=min_p).median()
    mad = (series - med).abs().rolling(window=window, min_periods=min_p).median()
    return (series - med) / (1.4826 * mad).replace(0, np.nan)


In [ ]:
def parse_m5_data(from_date: str = '2014-01-01') -> pd.DataFrame:
    base_url = 'https://www.cbr.ru/hd_base/bliquidity/'
    from_dt = pd.Timestamp(from_date)
    to_dt = pd.Timestamp(datetime.now().date())

    url = (
        f'{base_url}?UniDbQuery.Posted=True'
        f'&UniDbQuery.From={from_dt:%d.%m.%Y}'
        f'&UniDbQuery.To={to_dt:%d.%m.%Y}'
    )

    s = make_session()
    html = s.get(url, timeout=60).text

    df = pd.read_html(io.StringIO(html), header=[0,1,2,3,4])[0]
    df.columns = flatten_columns(df.columns)

    if df.shape[1] < 15:
        raise RuntimeError(f'Unexpected CBR table shape: {df.shape}')

    date_col = df.columns[0]
    col_def = df.columns[1]
    col_def_wo = df.columns[2]
    col_corr = df.columns[13]
    col_res = df.columns[14]

    df = df[df[date_col].astype(str).str.contains(r'\d{2}\.\d{2}\.\d{4}', regex=True)].copy()
    df[date_col] = pd.to_datetime(df[date_col], dayfirst=True, errors='coerce')
    df = df.dropna(subset=[date_col]).sort_values(date_col).reset_index(drop=True)

    for c in [col_def, col_def_wo, col_corr, col_res]:
        df[c] = to_num(df[c])

    m5 = pd.DataFrame({
        'date': df[date_col],
        'liquidity_deficit': df[col_def],
        'liquidity_deficit_ex_corr': df[col_def_wo],
        'corr_accounts': df[col_corr],
        'required_reserves': df[col_res],
    })

    m5 = m5[m5['date'] >= from_dt].copy()
    m5['reserve_buffer'] = m5['corr_accounts'] - m5['required_reserves']

    # ??????-???????? ??????????? ??? M5
    m5['treasury_pressure'] = m5['liquidity_deficit_ex_corr'].diff()
    m5['treasury_pressure_ma7'] = m5['treasury_pressure'].rolling(7, min_periods=3).mean()

    # MAD-???????????? (?????????? ???? ~3 ???? ???????? ????)
    m5['MAD_score_treasury_pressure'] = mad_score(m5['treasury_pressure'], window=756)
    m5['MAD_score_liquidity_deficit'] = mad_score(m5['liquidity_deficit'], window=756)

    m5['is_month_end'] = m5['date'].dt.is_month_end.astype(int)
    m5['flag_stress'] = ((m5['MAD_score_treasury_pressure'] > 2.5) |
                         ((m5['MAD_score_treasury_pressure'] > 1.5) & (m5['is_month_end'] == 1))).astype(int)

    return m5.reset_index(drop=True)


In [ ]:
def plot_m5(df: pd.DataFrame, out_file: str | None = None) -> None:
    fig, axes = plt.subplots(3, 1, figsize=(14, 11), sharex=True)

    axes[0].plot(df['date'], df['liquidity_deficit'], label='Liquidity deficit', color='#1f77b4')
    axes[0].plot(df['date'], df['liquidity_deficit_ex_corr'], label='Deficit ex corr', color='#ff7f0e', alpha=0.9)
    axes[0].set_title('M5: Liquidity Deficit Dynamics')
    axes[0].legend()
    axes[0].grid(alpha=0.25)

    axes[1].plot(df['date'], df['treasury_pressure'], label='Treasury pressure (d/d)', color='#2ca02c', alpha=0.8)
    axes[1].plot(df['date'], df['treasury_pressure_ma7'], label='MA7', color='#d62728', linewidth=2)
    axes[1].set_title('M5: Treasury Pressure Proxy')
    axes[1].legend()
    axes[1].grid(alpha=0.25)

    axes[2].plot(df['date'], df['MAD_score_treasury_pressure'], label='MAD score (pressure)', color='#9467bd')
    axes[2].axhline(2.5, color='red', linestyle='--', linewidth=1, label='Stress threshold')
    stress = df[df['flag_stress'] == 1]
    axes[2].scatter(stress['date'], stress['MAD_score_treasury_pressure'], color='red', s=18, label='Flag stress')
    axes[2].set_title('M5: Stress Signal')
    axes[2].legend()
    axes[2].grid(alpha=0.25)

    plt.tight_layout()
    if out_file:
        out_path = Path(out_file)
        out_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(out_path, dpi=150)
    plt.show()


In [ ]:
# ?????? M5
m5_df = parse_m5_data(from_date='2014-01-01')
m5_df.tail()


### EDA: ликвидность и стресс-сигналы M5

Размер выборки, пропуски, распределения ключевых метрик и доля дней с `flag_stress`.

In [ ]:
print(m5_df.info())
print("\nПериод:", m5_df["date"].min().date(), "—", m5_df["date"].max().date())
num = m5_df.select_dtypes(include=[np.number]).columns
print("\nПропуски (% строк):")
print((m5_df[num].isna().mean() * 100).round(2).sort_values(ascending=False))

core = [
    "liquidity_deficit",
    "liquidity_deficit_ex_corr",
    "corr_accounts",
    "required_reserves",
    "reserve_buffer",
    "treasury_pressure",
    "MAD_score_treasury_pressure",
    "MAD_score_liquidity_deficit",
]
avail = [c for c in core if c in m5_df.columns]
display(m5_df[avail].describe().T)

print("\nСтресс-флаг по годам:")
print(m5_df.assign(y=m5_df["date"].dt.year).groupby("y")["flag_stress"].agg(["sum", "mean"]).round(3))

fig, axes = plt.subplots(2, 2, figsize=(11, 7))
axes = axes.ravel()

def _hist(ax, series, title):
    s = series.dropna()
    ax.hist(s, bins=min(40, max(15, len(s) // 50)), color="steelblue", edgecolor="white")
    ax.set_title(title)

if "liquidity_deficit" in m5_df.columns:
    _hist(axes[0], m5_df["liquidity_deficit"], "liquidity_deficit")
if "treasury_pressure" in m5_df.columns:
    _tp = m5_df["treasury_pressure"].dropna()
    _lo, _hi = _tp.quantile(0.01), _tp.quantile(0.99)
    _hist(axes[1], _tp.clip(_lo, _hi), "treasury_pressure (1–99% квантиль)")
if "reserve_buffer" in m5_df.columns:
    _hist(axes[2], m5_df["reserve_buffer"], "reserve_buffer")
if "MAD_score_treasury_pressure" in m5_df.columns:
    _hist(axes[3], m5_df["MAD_score_treasury_pressure"].clip(-5, 5), "MAD pressure (±5)")
plt.tight_layout()
plt.show()

if len(avail) >= 3:
    cm = m5_df[avail].corr(numeric_only=True)
    fig2, ax2 = plt.subplots(figsize=(9, 7))
    im = ax2.imshow(cm, cmap="coolwarm", vmin=-1, vmax=1, aspect="auto")
    plt.colorbar(im, ax=ax2, shrink=0.6)
    ax2.set_xticks(range(len(cm.columns)))
    ax2.set_xticklabels(cm.columns, rotation=45, ha="right")
    ax2.set_yticks(range(len(cm.columns)))
    ax2.set_yticklabels(cm.columns)
    ax2.set_title("Корреляция (Pearson)")
    plt.tight_layout()
    plt.show()

In [ ]:
# ????????????
plot_m5(m5_df, out_file='outputs/m5/m5_dashboard.png')


In [ ]:
# ?????????? ???????
Path('outputs/m5').mkdir(parents=True, exist_ok=True)
m5_df.to_csv('outputs/m5/m5_dataset.csv', index=False, encoding='utf-8-sig')

print('Rows:', len(m5_df))
print('Date range:', m5_df['date'].min(), '->', m5_df['date'].max())
print('Stress flags:', int(m5_df['flag_stress'].sum()))
print('Saved: outputs/m5/m5_dataset.csv')
print('Saved: outputs/m5/m5_dashboard.png')
